In [1]:
import numpy as np  
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix



In [2]:
df = pd.read_csv('MedTech_Masterchart - final (1).csv')

In [3]:
df.shape

(226, 329)

In [4]:
df.head()

,S.no,Age,weight,height,Pulse rate,Pain,Site,staging (TNM),Vata,Pitta,...,K_Q62.2,K_Q63,K_Q64,K_Q65_final,K_Q65,K_Q65.1,K_Q66_final,K_Q66,K_Q66.1,K_Q67
0,1,60,75,168,73,5,Hypopharyngeal,T2N2cM0,25.2,24.4,...,0.5,1,1,1,0,1,0,0,0,1
1,2,61,65,165,91,6,Glottis,T3N1M0,38.2,37.9,...,0.0,0,0,1,0,1,1,0,1,0
2,3,56,70,170,88,2,Glottis,T2N0M0,29.1,32.2,...,0.5,1,1,1,0,1,0,0,0,0
3,4,46,67,185,72,7,Oropharyngeal,T4bN2cM0,29.5,24.0,...,0.5,1,1,1,0,1,0,0,0,1
4,5,63,67,185,85,3,Oropharyngeal,T2N0M0,38.9,21.7,...,0.5,1,1,1,0,1,0,0,0,1


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 226 entries, 0 to 225
Columns: 329 entries, S.no to K_Q67
dtypes: float64(54), int64(273), object(2)
memory usage: 581.0+ KB


In [6]:
print(df.isnull().sum().sort_values(ascending=False).head(30))


P_Q16          141
P_Q19          105
V_Q9            61
P_Q9            61
V_Q18           55
K_Q4            11
V_Q4            11
P_Q13           10
K_Q13           10
P_Q56.3          0
P_Q59            0
P_Q59_final      0
P_Q58            0
P_Q57            0
P_Q56.4          0
P_Q56            0
P_Q56.2          0
P_Q56.1          0
P_Q59.2          0
P_Q56_final      0
P_Q55            0
P_Q54            0
P_Q53.1          0
P_Q53            0
P_Q59.1          0
P_Q62_final      0
P_Q60            0
P_Q61            0
K_Q3             0
K_Q2.1           0
dtype: int64


In [7]:
# We'll handle missing values inside model pipelines to avoid leakage.
print('Missing values (top 30):')
print(df.isnull().sum().sort_values(ascending=False).head(30))


Missing values (top 30):
P_Q16          141
P_Q19          105
V_Q9            61
P_Q9            61
V_Q18           55
K_Q4            11
V_Q4            11
P_Q13           10
K_Q13           10
P_Q56.3          0
P_Q59            0
P_Q59_final      0
P_Q58            0
P_Q57            0
P_Q56.4          0
P_Q56            0
P_Q56.2          0
P_Q56.1          0
P_Q59.2          0
P_Q56_final      0
P_Q55            0
P_Q54            0
P_Q53.1          0
P_Q53            0
P_Q59.1          0
P_Q62_final      0
P_Q60            0
P_Q61            0
K_Q3             0
K_Q2.1           0
dtype: int64


In [8]:
# Get all column names
cols = df.columns

# Columns to keep
keep_cols = []

for col in cols:
    # If it's a "_final" column → always keep
    if col.endswith('_final'):
        keep_cols.append(col)
    else:
        # Check if corresponding _final exists
        base = col.split('.')[0]  # handles K_Q65.1
        final_col = base + '_final'
        
        if final_col not in cols:
            keep_cols.append(col)

# Keep only selected columns
df = df[keep_cols]

In [9]:
df.head()

,S.no,Age,weight,height,Pulse rate,Pain,Site,staging (TNM),Vata,Pitta,...,K_Q58,K_Q59_final,K_Q60,K_Q61,K_Q62_final,K_Q63,K_Q64,K_Q65_final,K_Q66_final,K_Q67
0,1,60,75,168,73,5,Hypopharyngeal,T2N2cM0,25.2,24.4,...,0.5,1,1,0.5,1.0,1,1,1,0,1
1,2,61,65,165,91,6,Glottis,T3N1M0,38.2,37.9,...,0.5,1,0,0.0,0.0,0,0,1,1,0
2,3,56,70,170,88,2,Glottis,T2N0M0,29.1,32.2,...,0.5,1,1,0.5,1.0,1,1,1,0,0
3,4,46,67,185,72,7,Oropharyngeal,T4bN2cM0,29.5,24.0,...,0.5,1,1,0.5,1.0,1,1,1,0,1
4,5,63,67,185,85,3,Oropharyngeal,T2N0M0,38.9,21.7,...,0.5,1,1,0.5,1.0,1,1,1,0,1


In [120]:
df.to_csv("data.csv", index=False)

In [10]:
df.shape

(226, 251)

In [11]:
print(df.columns.tolist())

['S.no', 'Age', 'weight', 'height', 'Pulse rate', 'Pain', 'Site', 'staging (TNM)', 'Vata', 'Pitta', 'Kapha', 'SF 36', 'Value_Q1', 'Value_Q2', 'Value_Q3', 'Value_Q4', 'Value_Q5', 'Value_Q6', 'Value_Q7', 'Value_Q8', 'Value_Q9', 'Value_Q10', 'Value_Q11', 'Value_Q12', 'Value_Q13', 'Value_Q14', 'Value_Q15', 'Value_Q16', 'Value_Q17', 'Value_Q18', 'Value_Q19', 'Value_Q20', 'Value_Q21', 'Value_Q22', 'Value_Q23', 'Value_Q24', 'Value_Q25', 'Value_Q26', 'Value_Q27', 'Value_Q28', 'Value_Q29', 'Value_Q30', 'Value_Q31', 'Value_Q32', 'Value_Q33', 'Value_Q34', 'Value_Q35', 'Value_Q36', 'V_Q1', 'V_Q2_final', 'V_Q3', 'V_Q4', 'V_Q5', 'V_Q6', 'V_Q7', 'V_Q8', 'V_Q9', 'V_Q10', 'V_Q11', 'V_Q12', 'V_Q13', 'V_Q14', 'V_Q15', 'V_Q16', 'V_Q17_final', 'V_Q18', 'V_Q19', 'V_Q20', 'V_Q21_final', 'V_Q22', 'V_Q23', 'V_Q24', 'V_Q25', 'V_Q26', 'V_Q27', 'V_Q28', 'V_Q29', 'V_Q30', 'V_Q31', 'V_Q32', 'V_Q33', 'V_Q34', 'V_Q35', 'V_Q36', 'V_Q37', 'V_Q38', 'V_Q39', 'V_Q40', 'V_Q41', 'V_Q42', 'V_Q43', 'V_Q44', 'V_Q45', 'V_Q46', 

In [12]:
for col in df.columns:
    print(f'{col}: {df[col].dtypes}')

S.no: int64
Age: int64
weight: int64
height: int64
Pulse rate: int64
Pain: int64
Site: object
staging (TNM): object
Vata: float64
Pitta: float64
Kapha: float64
SF 36: int64
Value_Q1: int64
Value_Q2: int64
Value_Q3: int64
Value_Q4: int64
Value_Q5: int64
Value_Q6: int64
Value_Q7: int64
Value_Q8: int64
Value_Q9: int64
Value_Q10: int64
Value_Q11: int64
Value_Q12: int64
Value_Q13: int64
Value_Q14: int64
Value_Q15: int64
Value_Q16: int64
Value_Q17: int64
Value_Q18: int64
Value_Q19: int64
Value_Q20: int64
Value_Q21: float64
Value_Q22: int64
Value_Q23: float64
Value_Q24: float64
Value_Q25: float64
Value_Q26: float64
Value_Q27: float64
Value_Q28: float64
Value_Q29: float64
Value_Q30: float64
Value_Q31: float64
Value_Q32: int64
Value_Q33: int64
Value_Q34: int64
Value_Q35: int64
Value_Q36: int64
V_Q1: int64
V_Q2_final: int64
V_Q3: int64
V_Q4: float64
V_Q5: int64
V_Q6: int64
V_Q7: int64
V_Q8: int64
V_Q9: float64
V_Q10: int64
V_Q11: int64
V_Q12: int64
V_Q13: int64
V_Q14: int64
V_Q15: int64
V_Q16: i

In [13]:
df['Site'].value_counts()

Site
Oral cavity                 62
Supraglottic                39
Glottic                     33
Hypopharyngeal              30
Oropharyngeal               23
Sinonasal carcinoma         10
Nasopharyngeal Carcinoma     8
Maxillary sinus              5
Sinonasal Carcinoma          4
Glottis                      3
Esophagus                    2
Esophageal                   1
Sinonasal chondrosarcoma     1
Oropharynx                   1
sinonasal carcinoma          1
Sinonasal Sarcoma            1
Nasopharyngeal carcinoma     1
Transglottic                 1
Name: count, dtype: int64

In [14]:
import re
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder

# -----------------------------
# TNM normalization
# -----------------------------
def normalize_stage_tnm(raw):
    if pd.isna(raw):
        return np.nan

    s = str(raw).upper()

    # Remove unwanted characters (keep only TNM-related)
    s = re.sub(r'[^TNM0-9ABCIS]', '', s)

    # Extract T, N, M
    t_match = re.search(r'T(IS|[0-4][ABC]?)', s)
    n_match = re.search(r'N([0-3])[ABC]?', s)
    m_match = re.search(r'M([01])', s)

    if not (t_match and n_match and m_match):
        return np.nan

    t_token = t_match.group(1).lower()
    n_token = n_match.group(1)
    m_token = m_match.group(1)

    return f"T{t_token}N{n_token}M{m_token}"


# -----------------------------
# Apply on dataset
# -----------------------------

df["stage_clean"] = df["staging (TNM)"].apply(normalize_stage_tnm)

# Keep only valid rows
df = df[df["stage_clean"].notna()].copy()

# -----------------------------
# Active classes in dataset
# -----------------------------
active_stages = sorted(df["stage_clean"].unique())

print("Active TNM classes in dataset:", len(active_stages))
print(active_stages[:10])

# -----------------------------
# Label Encoding
# -----------------------------
stage_le = LabelEncoder()
y_stage = stage_le.fit_transform(df["stage_clean"])

print("\nEncoded classes:", len(stage_le.classes_))
print(stage_le.classes_[:10])

Active TNM classes in dataset: 21
['T1N1M0', 'T1aN0M0', 'T1bN0M0', 'T2N0M0', 'T2N1M0', 'T2N2M0', 'T2N3M0', 'T3N0M0', 'T3N1M0', 'T3N2M0']

Encoded classes: 21
['T1N1M0' 'T1aN0M0' 'T1bN0M0' 'T2N0M0' 'T2N1M0' 'T2N2M0' 'T2N3M0' 'T3N0M0'
 'T3N1M0' 'T3N2M0']


In [15]:
df


,S.no,Age,weight,height,Pulse rate,Pain,Site,staging (TNM),Vata,Pitta,...,K_Q59_final,K_Q60,K_Q61,K_Q62_final,K_Q63,K_Q64,K_Q65_final,K_Q66_final,K_Q67,stage_clean
0,1,60,75,168,73,5,Hypopharyngeal,T2N2cM0,25.2,24.4,...,1,1,0.5,1.0,1,1,1,0,1,T2N2M0
1,2,61,65,165,91,6,Glottis,T3N1M0,38.2,37.9,...,1,0,0.0,0.0,0,0,1,1,0,T3N1M0
2,3,56,70,170,88,2,Glottis,T2N0M0,29.1,32.2,...,1,1,0.5,1.0,1,1,1,0,0,T2N0M0
3,4,46,67,185,72,7,Oropharyngeal,T4bN2cM0,29.5,24.0,...,1,1,0.5,1.0,1,1,1,0,1,T4bN2M0
4,5,63,67,185,85,3,Oropharyngeal,T2N0M0,38.9,21.7,...,1,1,0.5,1.0,1,1,1,0,1,T2N0M0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
221,222,61,75,176,76,3,Hypopharyngeal,T3N0M0,29.5,24.0,...,1,1,0.5,1.0,1,1,1,0,1,T3N0M0
222,223,51,69,175,71,3,Oral cavity,T4aN2bM0,29.3,31.0,...,1,1,0.5,1.0,1,1,1,0,0,T4aN2M0
223,224,70,61,166,77,5,Glottic,T3N0M0,38.9,21.7,...,1,1,0.5,1.0,1,1,1,0,1,T3N0M0
224,225,75,52,176,74,3,Oropharyngeal,T3N1M0,36.6,23.9,...,1,1,0.5,1.0,1,1,1,0,1,T3N1M0


In [16]:
y_stage

array([ 5,  8,  3, 20,  3,  4,  2,  8,  8,  8,  9, 11,  8,  7,  8, 18,  7,
        8, 16, 19,  8,  8, 19,  5,  7,  6,  8,  8,  8, 18,  8, 16,  7,  1,
       12,  8, 18,  8, 13,  3, 18,  5, 16,  8,  7,  8, 18,  8,  7, 18,  3,
        9,  8,  8,  8, 18,  7, 16, 18,  0,  8,  8,  8,  4, 16,  2,  7,  8,
        7,  6,  8,  8,  8,  7,  7,  4,  8,  8, 18,  4, 18,  8,  9, 16,  8,
        8, 18,  8, 18, 18,  8,  8,  4,  7,  8,  7, 16,  8, 10,  6,  8,  6,
       18, 16,  7,  8,  8,  8,  8,  5,  4,  7, 18,  8, 10,  9,  8, 18,  8,
       18,  8,  8, 14,  7,  8,  7,  8,  8,  8,  8,  7,  8, 10,  8,  8,  8,
       17,  7,  5, 17,  4,  8,  8,  5,  8,  7, 16,  8,  8, 16, 18,  6, 16,
        5, 17,  8,  5, 18,  7,  5,  8,  8,  7,  9,  7,  8,  3,  8,  8,  7,
        8,  7, 18, 16,  8,  8,  9,  8, 18,  8,  8, 18,  7,  8, 18,  8,  8,
        4,  8,  7,  8,  8,  8,  8,  2, 15,  7,  6, 18,  4,  8,  8,  5,  8,
        7, 17,  8,  8, 12, 16,  6, 16,  5, 17,  8,  5, 18,  7,  5,  8,  9,
        7, 18,  7,  8,  3

In [17]:
import re
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder, MinMaxScaler


# -----------------------------
# 3. Filter valid rows
# -----------------------------
numeric_cols = ["Pain", "weight", "height", "Pulse rate"]

df[numeric_cols] = df[numeric_cols].apply(pd.to_numeric, errors="coerce")

mask = df["stage_clean"].notna()
for col in numeric_cols:
    mask &= df[col].notna()

df = df.loc[mask].copy()

# -----------------------------
# 4. Encode TNM (replace original column)
# -----------------------------
stage_le = LabelEncoder()
df["staging (TNM)"] = stage_le.fit_transform(df["stage_clean"])

print("Number of stage classes:", len(stage_le.classes_))
print("Sample classes:", stage_le.classes_[:10])

# -----------------------------
# 5. Min-Max scaling (replace original columns)
# -----------------------------
scaler = MinMaxScaler()
df[numeric_cols] = scaler.fit_transform(df[numeric_cols])

# -----------------------------
# 6. Cleanup (remove helper column)
# -----------------------------
df.drop(columns=["stage_clean"], inplace=True)

# -----------------------------
# Final dataset ready
# -----------------------------
print("\nFinal shape:", df.shape)
print(df.head())

Number of stage classes: 21
Sample classes: ['T1N1M0' 'T1aN0M0' 'T1bN0M0' 'T2N0M0' 'T2N1M0' 'T2N2M0' 'T2N3M0' 'T3N0M0'
 'T3N1M0' 'T3N2M0']

Final shape: (226, 251)
   S.no  Age    weight    height  Pulse rate   Pain            Site  \
0     1   60  0.681818  0.387097    0.343750  0.500  Hypopharyngeal   
1     2   61  0.454545  0.290323    0.625000  0.625         Glottis   
2     3   56  0.568182  0.451613    0.578125  0.125         Glottis   
3     4   46  0.500000  0.935484    0.328125  0.750   Oropharyngeal   
4     5   63  0.500000  0.935484    0.531250  0.250   Oropharyngeal   

   staging (TNM)  Vata  Pitta  ...  K_Q58  K_Q59_final  K_Q60  K_Q61  \
0              5  25.2   24.4  ...    0.5            1      1    0.5   
1              8  38.2   37.9  ...    0.5            1      0    0.0   
2              3  29.1   32.2  ...    0.5            1      1    0.5   
3             20  29.5   24.0  ...    0.5            1      1    0.5   
4              3  38.9   21.7  ...    0.5        

In [18]:
from sklearn.preprocessing import OneHotEncoder
import pandas as pd

# ensure df is DataFrame
print(type(df))

encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')

encoded = encoder.fit_transform(df[["Site"]])

encoded_df = pd.DataFrame(
    encoded,
    columns=encoder.get_feature_names_out(["Site"])
)

df = pd.concat([df, encoded_df], axis=1)
df.drop("Site", axis=1, inplace=True)

<class 'pandas.core.frame.DataFrame'>


In [19]:
df

,S.no,Age,weight,height,Pulse rate,Pain,staging (TNM),Vata,Pitta,Kapha,...,Site_Oral cavity,Site_Oropharyngeal,Site_Oropharynx,Site_Sinonasal Carcinoma,Site_Sinonasal Sarcoma,Site_Sinonasal carcinoma,Site_Sinonasal chondrosarcoma,Site_Supraglottic,Site_Transglottic,Site_sinonasal carcinoma
0,1,60,0.681818,0.387097,0.343750,0.500,5,25.2,24.4,50.4,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2,61,0.454545,0.290323,0.625000,0.625,8,38.2,37.9,23.9,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,3,56,0.568182,0.451613,0.578125,0.125,3,29.1,32.2,38.8,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,4,46,0.500000,0.935484,0.328125,0.750,20,29.5,24.0,46.5,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,5,63,0.500000,0.935484,0.531250,0.250,3,38.9,21.7,39.3,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
221,222,61,0.681818,0.645161,0.390625,0.250,7,29.5,24.0,46.5,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
222,223,51,0.545455,0.612903,0.312500,0.250,18,29.3,31.0,39.7,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
223,224,70,0.363636,0.322581,0.406250,0.500,7,38.9,21.7,39.3,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
224,225,75,0.159091,0.645161,0.359375,0.250,8,36.6,23.9,39.5,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [20]:
import pandas as pd
import re

def prepare_dataset(df, mode="value"):
    """
    mode options:
    - "value"     → Value_Q1 to Value_Q36
    - "vpk"       → V_Q*, P_Q*, K_Q*
    - "all"       → Value_Q* + V_Q* + P_Q* + K_Q* + Site_*

    Returns:
      X      : feature matrix
      y_stg  : staging (TNM)
      y_pain : Pain
    """

    df = df.copy()

    # --- 1. Clean column names ---
    df.columns = df.columns.str.strip()
    df.columns = [re.sub(r'\.\d+$', '', col) for col in df.columns]
    df.columns = [col.replace('-', '_') for col in df.columns]
    df = df.loc[:, ~df.columns.duplicated()]

    # --- 2. Targets ---
    if "staging (TNM)" not in df.columns:
        raise ValueError("Target column 'staging (TNM)' not found")
    if "Pain" not in df.columns:
        raise ValueError("Target column 'Pain' not found")

    y_stg = pd.to_numeric(df["staging (TNM)"], errors="coerce")
    y_pain = pd.to_numeric(df["Pain"], errors="coerce")

    # keep only rows where both targets are valid
    mask = y_stg.notna() & y_pain.notna()
    df = df.loc[mask].copy()
    y_stg = y_stg.loc[mask].astype(int)
    y_stg = y_stg.astype(int)
    y_pain = y_pain.loc[mask].astype(float)

    # --- 3. Select feature columns based on mode ---
    if mode == "value":
        feature_cols = [col for col in df.columns if col.startswith("Value_Q")]

    elif mode == "vpk":
        feature_cols = [
            col for col in df.columns
            if col.startswith("V_Q") or col.startswith("P_Q") or col.startswith("K_Q")
        ]

    elif mode == "all":
        feature_cols = [
            col for col in df.columns
            if col.startswith("Value_Q")
            or col.startswith("V_Q")
            or col.startswith("P_Q")
            or col.startswith("K_Q")
        ]

    else:
        raise ValueError("Invalid mode. Choose from: value, vpk, all")

    # --- 4. Add existing Site_ one-hot columns ---
    site_cols = [col for col in df.columns if col.startswith("Site_")]

    # --- 5. Final feature matrix ---
    final_cols = feature_cols + site_cols
    X = (
        df[final_cols]
        .apply(pd.to_numeric, errors="coerce")
        .replace([np.inf, -np.inf], np.nan)
        .fillna(0.0)
    )

    return (
        np.nan_to_num(np.asarray(X, dtype=np.float32), nan=0.0, posinf=0.0, neginf=0.0),
        np.asarray(y_stg),
        np.asarray(y_pain),
    )


# Example usage
X1, y_stg1, y_pain1 = prepare_dataset(df, mode="value")
X2, y_stg2, y_pain2 = prepare_dataset(df, mode="vpk")
X3, y_stg3, y_pain3 = prepare_dataset(df, mode="all")

print(X1.shape, X2.shape, X3.shape)
print(len(np.unique(y_stg1)), y_pain1.min(), y_pain1.max())

(226, 54) (226, 220) (226, 256)
21 0.0 1.0


In [23]:
import numpy as np
unique, counts = np.unique(y_stg1, return_counts=True)
print(dict(zip(unique, counts)))

{0: 1, 1: 1, 2: 3, 3: 6, 4: 9, 5: 13, 6: 7, 7: 32, 8: 90, 9: 7, 10: 3, 11: 1, 12: 2, 13: 1, 14: 1, 15: 1, 16: 14, 17: 5, 18: 26, 19: 2, 20: 1}


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

def compute_metrics(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    return {
        'MAE': mean_absolute_error(y_true, y_pred),
        'MSE': mse,
        'RMSE': np.sqrt(mse),
        'R2': r2_score(y_true, y_pred)
    }

def print_metrics(name, metrics):
    print(
        f"{name} | MAE: {metrics['MAE']:.4f} | MSE: {metrics['MSE']:.4f} | "
        f"RMSE: {metrics['RMSE']:.4f} | R2: {metrics['R2']:.4f}"
    )



In [198]:
import numpy as np
from collections import Counter

def fix_single_sample_classes(X, y_stg, y_pain, min_count=3):
    """
    Ensures each class in y_stg has at least `min_count` samples
    by duplicating rows.

    Parameters:
        X       : feature matrix (numpy array)
        y_stg   : class labels (numpy array)
        y_pain  : regression target (numpy array)
        min_count : minimum samples required per class

    Returns:
        X_new, y_stg_new, y_pain_new
    """

    X = np.asarray(X)
    y_stg = np.asarray(y_stg)
    y_pain = np.asarray(y_pain)

    class_counts = Counter(y_stg)

    X_list = [X]
    ystg_list = [y_stg]
    ypain_list = [y_pain]

    for cls, count in class_counts.items():
        if count < min_count:
            needed = min_count - count

            # indices of this class
            idx = np.where(y_stg == cls)[0]

            # randomly duplicate existing rows
            dup_idx = np.random.choice(idx, size=needed, replace=True)

            X_list.append(X[dup_idx])
            ystg_list.append(y_stg[dup_idx])
            ypain_list.append(y_pain[dup_idx])

    # combine all
    X_new = np.vstack(X_list)
    y_stg_new = np.concatenate(ystg_list)
    y_pain_new = np.concatenate(ypain_list)

    return X_new, y_stg_new, y_pain_new

In [207]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, mean_absolute_error, mean_squared_error

# ---------------------------------------------------
# 1) Device
# ---------------------------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# ---------------------------------------------------
# 2) Model
# ---------------------------------------------------
class MultiTaskNet(nn.Module):
    def __init__(self, input_dim, n_classes, dropout=0.10):
        super().__init__()

        self.shared = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(32, 16),
            nn.ReLU()
        )

        self.stage_head = nn.Sequential(
            nn.Linear(16, 16),
            nn.ReLU(),
            nn.Linear(16, n_classes)
        )

        self.pain_head = nn.Sequential(
            nn.Linear(16, 16),
            nn.ReLU(),
            nn.Linear(16, 1)
        )

    def forward(self, x):
        shared = self.shared(x)
        stage_out = self.stage_head(shared)
        pain_out = self.pain_head(shared)
        return stage_out, pain_out


def build_model(input_dim, n_classes, dropout=0.30):
    model = MultiTaskNet(input_dim=input_dim, n_classes=n_classes, dropout=dropout)
    return model.to(device)

# ---------------------------------------------------
# 3) Train + Evaluate
# ---------------------------------------------------
def run_experiment(X, y_stg, y_pain, dataset_name="dataset", random_state=42,
                   batch_size=16, epochs=2000, lr=1e-3,
                   weight_decay=1e-4, stage_loss_w=0.7, pain_loss_w=0.7):
    X, y_stg, y_pain = fix_single_sample_classes(X, y_stg, y_pain, min_count=3)

    X = np.asarray(X, dtype=np.float32)
    y_stg = np.asarray(y_stg, dtype=np.int64).reshape(-1)
    y_pain = np.asarray(y_pain, dtype=np.float32).reshape(-1, 1)

    n_classes = len(np.unique(y_stg))

    # Split: train/test stratified on stage
    X_train, X_test, ystg_train, ystg_test, ypain_train, ypain_test = train_test_split(
        X, y_stg, y_pain,
        test_size=0.20,
        random_state=random_state,
        stratify=y_stg
    )

    # Split train/val stratified
    X_train, X_val, ystg_train, ystg_val, ypain_train, ypain_val = train_test_split(
        X_train, ystg_train, ypain_train,
        test_size=0.20,
        random_state=random_state,
        stratify=ystg_train
    )

    # Tensors
    X_train_t = torch.tensor(X_train, dtype=torch.float32)
    ystg_train_t = torch.tensor(ystg_train, dtype=torch.long)
    ypain_train_t = torch.tensor(ypain_train, dtype=torch.float32)

    X_val_t = torch.tensor(X_val, dtype=torch.float32)
    ystg_val_t = torch.tensor(ystg_val, dtype=torch.long)
    ypain_val_t = torch.tensor(ypain_val, dtype=torch.float32)

    X_test_t = torch.tensor(X_test, dtype=torch.float32)
    ystg_test_t = torch.tensor(ystg_test, dtype=torch.long)
    ypain_test_t = torch.tensor(ypain_test, dtype=torch.float32)

    train_ds = TensorDataset(X_train_t, ystg_train_t, ypain_train_t)
    val_ds = TensorDataset(X_val_t, ystg_val_t, ypain_val_t)
    test_ds = TensorDataset(X_test_t, ystg_test_t, ypain_test_t)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False)

    model = build_model(input_dim=X.shape[1], n_classes=n_classes)

    stage_criterion = nn.CrossEntropyLoss()
    pain_criterion = nn.SmoothL1Loss()   # robust on small data
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

    scaler = torch.cuda.amp.GradScaler(enabled=(device.type == "cuda"))

    best_val_loss = float("inf")
    best_state = None
    patience = 20
    patience_ctr = 0

    for epoch in range(epochs):
        # ---------- train ----------
        model.train()
        train_loss = 0.0

        for xb, ysb, ypb in train_loader:
            xb = xb.to(device)
            ysb = ysb.to(device)
            ypb = ypb.to(device)

            optimizer.zero_grad(set_to_none=True)

            with torch.cuda.amp.autocast(enabled=(device.type == "cuda")):
                stage_logits, pain_pred = model(xb)
                pain_pred = pain_pred.squeeze(-1)

                loss_stage = stage_criterion(stage_logits, ysb)
                loss_pain = pain_criterion(pain_pred, ypb.squeeze(-1))
                loss = stage_loss_w * loss_stage + pain_loss_w * loss_pain

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            train_loss += loss.item() * xb.size(0)

        train_loss /= len(train_loader.dataset)

        # ---------- validation ----------
        model.eval()
        val_loss = 0.0

        with torch.no_grad():
            for xb, ysb, ypb in val_loader:
                xb = xb.to(device)
                ysb = ysb.to(device)
                ypb = ypb.to(device)

                stage_logits, pain_pred = model(xb)
                pain_pred = pain_pred.squeeze(-1)

                loss_stage = stage_criterion(stage_logits, ysb)
                loss_pain = pain_criterion(pain_pred, ypb.squeeze(-1))
                loss = stage_loss_w * loss_stage + pain_loss_w * loss_pain

                val_loss += loss.item() * xb.size(0)

        val_loss /= len(val_loader.dataset)

        # early stopping
        if val_loss < best_val_loss - 1e-6:
            best_val_loss = val_loss
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            patience_ctr = 0
        else:
            patience_ctr += 1

        if patience_ctr >= patience:
            break

    # restore best model
    if best_state is not None:
        model.load_state_dict(best_state)

    # ---------- test ----------
    model.eval()
    all_stage_true, all_stage_pred = [], []
    all_pain_true, all_pain_pred = [], []

    with torch.no_grad():
        for xb, ysb, ypb in test_loader:
            xb = xb.to(device)

            stage_logits, pain_pred = model(xb)
            stage_pred = torch.argmax(stage_logits, dim=1).cpu().numpy()
            pain_pred = pain_pred.squeeze(-1).cpu().numpy()

            all_stage_true.extend(ysb.numpy().tolist())
            all_stage_pred.extend(stage_pred.tolist())
            all_pain_true.extend(ypb.squeeze(-1).numpy().tolist())
            all_pain_pred.extend(pain_pred.tolist())

    all_stage_true = np.array(all_stage_true)
    all_stage_pred = np.array(all_stage_pred)
    all_pain_true = np.array(all_pain_true)
    all_pain_pred = np.array(all_pain_pred)

    stage_acc = accuracy_score(all_stage_true, all_stage_pred)
    stage_bal_acc = balanced_accuracy_score(all_stage_true, all_stage_pred)
    stage_f1_macro = f1_score(all_stage_true, all_stage_pred, average="macro")
    pain_mae = mean_absolute_error(all_pain_true, all_pain_pred)
    pain_rmse = mean_squared_error(all_pain_true, all_pain_pred)

    print(f"\n=== {dataset_name.upper()} ===")
    print("Input shape:", X.shape)
    print("Classes:", n_classes)
    print("Best val loss:", round(best_val_loss, 4))
    print("Test stage accuracy:", round(stage_acc, 4))
    print("Test stage balanced accuracy:", round(stage_bal_acc, 4))
    print("Test stage macro F1:", round(stage_f1_macro, 4))
    print("Test pain MAE:", round(pain_mae, 4))
    print("Test pain RMSE:", round(pain_rmse, 4))

    return {
        "model": model,
        "metrics": {
            "best_val_loss": best_val_loss,
            "stage_acc": stage_acc,
            "stage_bal_acc": stage_bal_acc,
            "stage_f1_macro": stage_f1_macro,
            "pain_mae": pain_mae,
            "pain_rmse": pain_rmse,
        }
    }

# ---------------------------------------------------
# 4) Run on your three datasets
# ---------------------------------------------------
# Assumes you already have:
# X1, y_stg1, y_pain1 = prepare_dataset(df, mode="value")
# X2, y_stg2, y_pain2 = prepare_dataset(df, mode="vpk")
# X3, y_stg3, y_pain3 = prepare_dataset(df, mode="all")

results_value = run_experiment(X1, y_stg1, y_pain1, dataset_name="value")
results_vpk   = run_experiment(X2, y_stg2, y_pain2, dataset_name="vpk")
results_all   = run_experiment(X3, y_stg3, y_pain3, dataset_name="all")

Using device: cuda


/tmp/ipykernel_68674/632099437.py:114: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(device.type == "cuda"))
/tmp/ipykernel_68674/632099437.py:133: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(device.type == "cuda")):
/home/yogendra/ml-dl/lib/python3.10/site-packages/sklearn/metrics/_classification.py:2801: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/tmp/ipykernel_68674/632099437.py:114: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(device.type == "cuda"))
/tmp/ipykernel_68674/632099437.py:133: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated


=== VALUE ===
Input shape: (242, 54)
Classes: 21
Best val loss: 1.4068
Test stage accuracy: 0.3878
Test stage balanced accuracy: 0.1296
Test stage macro F1: 0.1008
Test pain MAE: 0.148
Test pain RMSE: 0.0379

=== VPK ===
Input shape: (242, 220)
Classes: 21
Best val loss: 1.5731
Test stage accuracy: 0.3673
Test stage balanced accuracy: 0.0556
Test stage macro F1: 0.0308
Test pain MAE: 0.2097
Test pain RMSE: 0.0803


/tmp/ipykernel_68674/632099437.py:114: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(device.type == "cuda"))
/tmp/ipykernel_68674/632099437.py:133: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(device.type == "cuda")):



=== ALL ===
Input shape: (242, 256)
Classes: 21
Best val loss: 1.6099
Test stage accuracy: 0.3673
Test stage balanced accuracy: 0.0556
Test stage macro F1: 0.0299
Test pain MAE: 0.1917
Test pain RMSE: 0.0617


In [204]:
model1 = run_experiment(X1, y_stg1, y_pain1, "value")


/tmp/ipykernel_68674/1599092794.py:114: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(device.type == "cuda"))
/tmp/ipykernel_68674/1599092794.py:133: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(device.type == "cuda")):



=== VALUE ===
Input shape: (242, 54)
Classes: 21
Best val loss: 2.02
Test stage accuracy: 0.3469
Test stage balanced accuracy: 0.1049
Test stage macro F1: 0.0579
Test pain MAE: 0.1383
Test pain RMSE: 0.0277


In [ ]:
model2 = run_experiment(X2, y_stg2, y_pain2, "vpk")

/home/yogendra/ml-dl/lib/python3.10/site-packages/sklearn/metrics/_classification.py:2801: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")



=== VPK ===
Input shape: torch.Size([226, 220])
Classes: 21
Best val loss: 4.654105186462402
Test stage accuracy: 0.2609
Balanced accuracy: 0.0891
F1 macro: 0.0556
Pain MAE: 0.5236
Pain RMSE: 0.571


In [ ]:
model3 = run_experiment(X3, y_stg3, y_pain3, "all")


=== ALL ===
Input shape: torch.Size([226, 256])
Classes: 21
Best val loss: 4.551298141479492
Test stage accuracy: 0.087
Balanced accuracy: 0.1299
F1 macro: 0.0501
Pain MAE: 0.3957
Pain RMSE: 0.4657


/home/yogendra/ml-dl/lib/python3.10/site-packages/sklearn/metrics/_classification.py:2801: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


In [24]:
from sklearn.ensemble import RandomForestClassifier

clf = RandomForestClassifier()
clf.fit(X1, y_stg1)
print(clf.score(X1, y_stg1))

0.8893805309734514


In [ ]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, mean_absolute_error, mean_squared_error

# ---------------------------------------------------
# 1) Device
# ---------------------------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# ---------------------------------------------------
# 2) Model
# ---------------------------------------------------
class MultiTaskNet(nn.Module):
    def __init__(self, input_dim, n_classes, dropout=0.30):
        super().__init__()

        self.shared = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(32, 16),
            nn.ReLU()
        )

        self.stage_head = nn.Sequential(
            nn.Linear(16, 16),
            nn.ReLU(),
            nn.Linear(16, n_classes)
        )

        self.pain_head = nn.Sequential(
            nn.Linear(16, 16),
            nn.ReLU(),
            nn.Linear(16, 1)
        )

    def forward(self, x):
        shared = self.shared(x)
        stage_out = self.stage_head(shared)
        pain_out = self.pain_head(shared)
        return stage_out, pain_out


def build_model(input_dim, n_classes, dropout=0.30):
    model = MultiTaskNet(input_dim=input_dim, n_classes=n_classes, dropout=dropout)
    return model.to(device)

# ---------------------------------------------------
# 3) Train + Evaluate
# ---------------------------------------------------
def run_experiment(X, y_stg, y_pain, dataset_name="dataset", random_state=42,
                   batch_size=16, epochs=200, lr=5e-4,
                   weight_decay=1e-4, stage_loss_w=1.0, pain_loss_w=0.5):

    X = np.asarray(X, dtype=np.float32)
    y_stg = np.asarray(y_stg, dtype=np.int64).reshape(-1)
    y_pain = np.asarray(y_pain, dtype=np.float32).reshape(-1, 1)

    n_classes = len(np.unique(y_stg))

    # Split: train/test stratified on stage
    X_train, X_test, ystg_train, ystg_test, ypain_train, ypain_test = train_test_split(
        X, y_stg, y_pain,
        test_size=0.20,
        random_state=random_state,
        stratify=y_stg
    )

    # Split train/val stratified
    X_train, X_val, ystg_train, ystg_val, ypain_train, ypain_val = train_test_split(
        X_train, ystg_train, ypain_train,
        test_size=0.20,
        random_state=random_state,
        stratify=ystg_train
    )

    # Tensors
    X_train_t = torch.tensor(X_train, dtype=torch.float32)
    ystg_train_t = torch.tensor(ystg_train, dtype=torch.long)
    ypain_train_t = torch.tensor(ypain_train, dtype=torch.float32)

    X_val_t = torch.tensor(X_val, dtype=torch.float32)
    ystg_val_t = torch.tensor(ystg_val, dtype=torch.long)
    ypain_val_t = torch.tensor(ypain_val, dtype=torch.float32)

    X_test_t = torch.tensor(X_test, dtype=torch.float32)
    ystg_test_t = torch.tensor(ystg_test, dtype=torch.long)
    ypain_test_t = torch.tensor(ypain_test, dtype=torch.float32)

    train_ds = TensorDataset(X_train_t, ystg_train_t, ypain_train_t)
    val_ds = TensorDataset(X_val_t, ystg_val_t, ypain_val_t)
    test_ds = TensorDataset(X_test_t, ystg_test_t, ypain_test_t)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False)

    model = build_model(input_dim=X.shape[1], n_classes=n_classes)

    stage_criterion = nn.CrossEntropyLoss()
    pain_criterion = nn.SmoothL1Loss()   # robust on small data
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    scaler = torch.cuda.amp.GradScaler(enabled=(device.type == "cuda"))

    best_val_loss = float("inf")
    best_state = None
    patience = 20
    patience_ctr = 0

    for epoch in range(epochs):
        # ---------- train ----------
        model.train()
        train_loss = 0.0

        for xb, ysb, ypb in train_loader:
            xb = xb.to(device)
            ysb = ysb.to(device)
            ypb = ypb.to(device)

            optimizer.zero_grad(set_to_none=True)

            with torch.cuda.amp.autocast(enabled=(device.type == "cuda")):
                stage_logits, pain_pred = model(xb)
                pain_pred = pain_pred.squeeze(-1)

                loss_stage = stage_criterion(stage_logits, ysb)
                loss_pain = pain_criterion(pain_pred, ypb.squeeze(-1))
                loss = stage_loss_w * loss_stage + pain_loss_w * loss_pain

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            train_loss += loss.item() * xb.size(0)

        train_loss /= len(train_loader.dataset)

        # ---------- validation ----------
        model.eval()
        val_loss = 0.0

        with torch.no_grad():
            for xb, ysb, ypb in val_loader:
                xb = xb.to(device)
                ysb = ysb.to(device)
                ypb = ypb.to(device)

                stage_logits, pain_pred = model(xb)
                pain_pred = pain_pred.squeeze(-1)

                loss_stage = stage_criterion(stage_logits, ysb)
                loss_pain = pain_criterion(pain_pred, ypb.squeeze(-1))
                loss = stage_loss_w * loss_stage + pain_loss_w * loss_pain

                val_loss += loss.item() * xb.size(0)

        val_loss /= len(val_loader.dataset)

        # early stopping
        if val_loss < best_val_loss - 1e-6:
            best_val_loss = val_loss
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            patience_ctr = 0
        else:
            patience_ctr += 1

        if patience_ctr >= patience:
            break

    # restore best model
    if best_state is not None:
        model.load_state_dict(best_state)

    # ---------- test ----------
    model.eval()
    all_stage_true, all_stage_pred = [], []
    all_pain_true, all_pain_pred = [], []

    with torch.no_grad():
        for xb, ysb, ypb in test_loader:
            xb = xb.to(device)

            stage_logits, pain_pred = model(xb)
            stage_pred = torch.argmax(stage_logits, dim=1).cpu().numpy()
            pain_pred = pain_pred.squeeze(-1).cpu().numpy()

            all_stage_true.extend(ysb.numpy().tolist())
            all_stage_pred.extend(stage_pred.tolist())
            all_pain_true.extend(ypb.squeeze(-1).numpy().tolist())
            all_pain_pred.extend(pain_pred.tolist())

    all_stage_true = np.array(all_stage_true)
    all_stage_pred = np.array(all_stage_pred)
    all_pain_true = np.array(all_pain_true)
    all_pain_pred = np.array(all_pain_pred)

    stage_acc = accuracy_score(all_stage_true, all_stage_pred)
    stage_bal_acc = balanced_accuracy_score(all_stage_true, all_stage_pred)
    stage_f1_macro = f1_score(all_stage_true, all_stage_pred, average="macro")
    pain_mae = mean_absolute_error(all_pain_true, all_pain_pred)
    pain_rmse = mean_squared_error(all_pain_true, all_pain_pred, squared=False)

    print(f"\n=== {dataset_name.upper()} ===")
    print("Input shape:", X.shape)
    print("Classes:", n_classes)
    print("Best val loss:", round(best_val_loss, 4))
    print("Test stage accuracy:", round(stage_acc, 4))
    print("Test stage balanced accuracy:", round(stage_bal_acc, 4))
    print("Test stage macro F1:", round(stage_f1_macro, 4))
    print("Test pain MAE:", round(pain_mae, 4))
    print("Test pain RMSE:", round(pain_rmse, 4))

    return {
        "model": model,
        "metrics": {
            "best_val_loss": best_val_loss,
            "stage_acc": stage_acc,
            "stage_bal_acc": stage_bal_acc,
            "stage_f1_macro": stage_f1_macro,
            "pain_mae": pain_mae,
            "pain_rmse": pain_rmse,
        }
    }

# ---------------------------------------------------
# 4) Run on your three datasets
# ---------------------------------------------------
# Assumes you already have:
# X1, y_stg1, y_pain1 = prepare_dataset(df, mode="value")
# X2, y_stg2, y_pain2 = prepare_dataset(df, mode="vpk")
# X3, y_stg3, y_pain3 = prepare_dataset(df, mode="all")

results_value = run_experiment(X1, y_stg1, y_pain1, dataset_name="value")
results_vpk   = run_experiment(X2, y_stg2, y_pain2, dataset_name="vpk")
results_all   = run_experiment(X3, y_stg3, y_pain3, dataset_name="all")